# Load libraries

In [30]:
from pathlib import Path
import pyvista as pv
import numpy as np
import trimesh
import pymeshlab
from pymeshlab import PercentageValue
import os
from stl import mesh
import manifold3d

# <span style="color:green"> Tubing specifications

In [31]:
filepath = 'xxx'

In [33]:
pores_filepath = filepath + '/cleaned/9_repaired.stl'
Specifications_file_path = filepath + '/User_Specifications.txt'

# Initialize the variables
cylinder_height = None
cylinder_radius = None

# Read the file and extract the values
with open(Specifications_file_path, 'r') as file:
    for line in file:
        if line.startswith("cylinder_height:"):
            # Extract the value after the colon and strip any whitespace
            cylinder_height = float(line.split(":")[1].strip())
        if line.startswith("cylinder_radius:"):
            # Extract the value after the colon and strip any whitespace
            cylinder_radius = float(line.split(":")[1].strip())


In [34]:
# Desired diameter and height of the cylinders
sandwiching_radius = cylinder_radius * 1.1
sandwiching_height = cylinder_height * 5

In [35]:
# position of the sadwiching cylinders (how deep in the model it is) by percentage height of the actual model (ex 10% = 10)
sand_per = 5

# percentage of inlet in the sandwiching cylinder to the total height of the sandwiching cylinder
in_per = 1

In [36]:
# Variables to save
Specifications_type = "Tubing_specifications"

# Combine variable names and values into a dictionary
specifications = {
    "\n\nSpecifications": Specifications_type,
    "sandwiching_cylinder_radius": sandwiching_radius,
    "sandwiching_cylinde_height": sandwiching_height,
    "position of the sadwiching cylinders (how deep in the model it is) by percentage height of the actual model (ex 10% = 10)": sand_per,
    "percentage of inlet in the sandwiching cylinder to the total height of the sandwiching cylinder": in_per,
}

# Open the file in write mode
with open(Specifications_file_path, "a") as file:
    # Write each variable as "variable_name: variable"
    for name, value in specifications.items():
        file.write(f"{name}: {value}\n")

### Sandwitch the model between 2 cylinders

### Create the 2 parts of each of the 2 sandwiching cylinders 

In [37]:
InOut = sand_per/100
in_c = in_per/100

# Creating one sandwiching cylinder (1 is the the cylinder deep in the model {sand_per} and 2 is the cylinder extrusion) all created at the center almost (0,0,0)
cylinder_1T = trimesh.primitives.Cylinder(radius = sandwiching_radius, height = sandwiching_height*(1-in_c), sections=100)
cylinder_2T = trimesh.primitives.Cylinder(radius = sandwiching_radius, height = sandwiching_height*(in_c), sections=100)

cylinder_1B = trimesh.primitives.Cylinder(radius = sandwiching_radius, height = sandwiching_height*(1-in_c), sections=100)
cylinder_2B = trimesh.primitives.Cylinder(radius = sandwiching_radius, height = sandwiching_height*(in_c), sections=100)



### Find the actaul height of the model and place the cylinders at the min and max (with some depth inside the model)

In [38]:
pores_mesh = trimesh.exchange.load.load(pores_filepath)
center_x, center_y, center_z = pores_mesh.centroid

# Get the vertex coordinates of the pores_mesh
vertices = pores_mesh.vertices

# Find the minimum and maximum Z-coordinate values
min_z = vertices[:, 2].min()
max_z = vertices[:, 2].max()

# Calculate the height range
height_range = max_z - min_z


In [39]:
# position of the sandwiching cylinders as percentage of the ROI cylinder height (location: 2T-1T-pores-1B-2B)
pos_1T = [center_x, center_y, center_z + cylinder_height/2 - cylinder_height*(InOut) + sandwiching_height*(1-in_c)/2]
pos_2T = [center_x, center_y, center_z + cylinder_height/2 - cylinder_height*(InOut) + sandwiching_height*(1-in_c) + sandwiching_height*(in_c)/2]

pos_1B = [center_x, center_y, center_z - cylinder_height/2 + cylinder_height*(InOut) - sandwiching_height*(1-in_c)/2]
pos_2B = [center_x, center_y, center_z - cylinder_height/2 + cylinder_height*(InOut) - sandwiching_height*(1-in_c) - sandwiching_height*(in_c)/2]


# Translate the cylinders to their respective positions:
cylinder_1T.apply_translation(pos_1T)
cylinder_2T.apply_translation(pos_2T)

cylinder_1B.apply_translation(pos_1B)
cylinder_2B.apply_translation(pos_2B)

# save:
cylinder_1T.export(filepath + '/cylinder_1T.stl', file_type='stl');
cylinder_2T.export(filepath + '/cylinder_2T.stl', file_type='stl');
cylinder_1B.export(filepath + '/cylinder_1B.stl', file_type='stl');
cylinder_2B.export(filepath + '/cylinder_2B.stl', file_type='stl');

### Make sure the stl parts are open to each other (Union combination)

In [41]:
# Load the STL files
mesh1 = trimesh.load_mesh(filepath + '/cylinder_1T.stl');

# Load the second STL file
mesh2 = trimesh.load_mesh(filepath + '/cylinder_2T.stl');

# Load the third STL file
mesh3 = trimesh.load_mesh(filepath + '/cylinder_1B.stl');

# Load the fourth STL file
mesh4 = trimesh.load_mesh(filepath + '/cylinder_2B.stl');

# Load the fifth STL file
mesh5 = trimesh.load_mesh(filepath + '/cleaned/9_repaired.stl');

# Combine the meshes
combined_mesh = trimesh.boolean.union([mesh1, mesh2, mesh3, mesh4, mesh5])

# Check if the meshes intersect
if combined_mesh.is_volume:
    # If they intersect, make sure they are open to each other
    combined_mesh.fix_normals()

# Save the combined mesh to a new STL file
combined_mesh.export(filepath + '/cleaned/10_sandwiched.stl');


# <span style="color:green"> Preparing model for OpenFoam: Splitting the model into 3 parts (bot, top, and wall) and removing binary encoding then rename inside file

### Import Data


In [ ]:
import_model_filepath = filepath + '/cleaned/10_sandwiched.stl'

In [11]:
# read the STL file and call it "mesh"
mesh = pv.read(import_model_filepath)

In [12]:
# read the mesh (here it still has no active arrays)
mesh

PolyData,Information
N Cells,659698
N Points,327395
X Bounds,"-3.299e+00, 3.301e+00"
Y Bounds,"-3.294e+00, 3.306e+00"
Z Bounds,"-5.450e+00, 5.450e+00"
N Arrays,0


In [13]:
# Take a screenshot of the model and save as .png without needing to show
plotter = pv.Plotter(off_screen=True)
plotter.add_mesh(mesh)
plotter.show(screenshot= filepath + "/myScreenshot.png")

C:\Users\Ahmad Alminnawi\AppData\Local\NA-MIC\Slicer 5.0.3\lib\Python\Lib\site-packages\pyvista\plotting\plotting.py:5523: UserWarning: 
Set `jupyter_backend` backend to `"none"` to take a screenshot within a notebook environment.
  warnings.warn(


ViewInteractiveWidget(height=768, layout=Layout(height='auto', width='100%'), width=1024)

### Threshold w.r.t cell centers to divide the file without losing cells

In [14]:
# get the center of all cells
centers = mesh.cell_centers()

In [15]:
# read the coordinates of the centers
centers.points

pyvista_ndarray([[-1.08184191,  0.1768573 ,  0.44963786],
                 [ 0.13870046, -1.87760631,  0.44963786],
                 [ 0.13700668, -1.87659291,  0.44963786],
                 ...,
                 [-2.48425754, -0.69096404, -0.02146506],
                 [-2.48762218, -0.68110281, -0.0119022 ],
                 [ 2.21046575, -0.02069273,  0.03846509]])

In [16]:
# only keep Z coordinates of the centers (read all arrays and only keep the last cell (2) which is Z)
mesh.cell_data['centerZ'] = centers.points[:,2]

In [17]:
# read the coordinates of the centers (now it has 1 active array which is the Z of each center)
mesh

PolyData (0x14517839c40)
  N Cells:	659698
  N Points:	327395
  X Bounds:	-3.299e+00, 3.301e+00
  Y Bounds:	-3.294e+00, 3.306e+00
  Z Bounds:	-5.450e+00, 5.450e+00
  N Arrays:	1

In [18]:
#calculate height of model to be split into top bot and wall according to the user input "sand_per" which is here "InOut"
min_value = min(mesh.active_scalars)
max_value = max(mesh.active_scalars)
height = max_value - min_value

#thresholding
thresh_min1 = min_value + sandwiching_height*(in_c)
thresh_min2 = min_value + sandwiching_height*(in_c) + sandwiching_height*(1-in_c)


thresh_max1 = max_value - sandwiching_height*(in_c)
thresh_max2 = max_value - sandwiching_height*(in_c) - sandwiching_height*(1-in_c)


#change min value to include more from bottom and max vlaue to include more from the top (not to lose any cells)
min_value = min_value - 1
max_value = max_value + 1


In [19]:
# Apply thresholding from min level to threshold of min level w.r.t. active array (centerZ) and take bottom
bot_mesh = mesh.threshold(value= (min_value, thresh_min1)).extract_surface()

In [20]:
# save the remaining part (tube_bot, wall, top, tube_top) 
remaining_mesh = mesh.threshold(value= (min_value, thresh_min1), invert=True).extract_surface()

In [21]:
# Apply thresholding from threshold of min to threshold of max level w.r.t. active array (centerZ) and take tube_bot
tube_bot_mesh = remaining_mesh.threshold(value= (thresh_min1, thresh_min2 )).extract_surface()

In [22]:
# save the remaining part (wall, top, tube_top) 
remaining_mesh = mesh.threshold(value= (thresh_min1, thresh_min2), invert=True).extract_surface()

In [23]:
# Apply thresholding from threshold of min to threshold of max level w.r.t. active array (centerZ) and take wall
wall_mesh = remaining_mesh.threshold(value= (thresh_min2, thresh_max2 )).extract_surface()

In [24]:
# save the remaining part (top, tube_top) 
remaining_mesh = mesh.threshold(value= (thresh_min2, thresh_max2), invert=True).extract_surface()

In [25]:
# Apply thresholding from threshold of min to threshold of max level w.r.t. active array (centerZ) and take tube_top
tube_top_mesh = remaining_mesh.threshold(value= (thresh_max2, thresh_max1 )).extract_surface()

In [26]:
# save the remaining part which is now the top  
remaining_mesh = remaining_mesh.threshold(value= (thresh_max2, thresh_max1 ), invert=True).extract_surface()

In [27]:
# Apply thresholding from threshold of min to threshold of max level w.r.t. active array (centerZ) and take tube_top
top_mesh = remaining_mesh.threshold(value= (thresh_max1, max_value )).extract_surface()

In [28]:
# read bottom results
bot_mesh

PolyData (0x144e5b64ca0)
  N Cells:	298
  N Points:	200
  X Bounds:	-3.299e+00, 3.301e+00
  Y Bounds:	-3.294e+00, 3.306e+00
  Z Bounds:	-5.450e+00, -5.400e+00
  N Arrays:	3

In [29]:
# read bottom results
tube_bot_mesh

PolyData (0x144e5b7d100)
  N Cells:	200
  N Points:	200
  X Bounds:	-3.299e+00, 3.301e+00
  Y Bounds:	-3.294e+00, 3.306e+00
  Z Bounds:	-5.400e+00, -4.504e-01
  N Arrays:	3

In [30]:
# read wall results
wall_mesh

PolyData (0x144e5b64d60)
  N Cells:	658702
  N Points:	326995
  X Bounds:	-3.299e+00, 3.301e+00
  Y Bounds:	-3.294e+00, 3.306e+00
  Z Bounds:	-4.504e-01, 4.496e-01
  N Arrays:	3

In [31]:
# read bottom results
tube_top_mesh

PolyData (0x144e5b7d700)
  N Cells:	200
  N Points:	200
  X Bounds:	-3.299e+00, 3.301e+00
  Y Bounds:	-3.294e+00, 3.306e+00
  Z Bounds:	4.496e-01, 5.400e+00
  N Arrays:	3

In [32]:
# read top results
top_mesh

PolyData (0x144e5b7d880)
  N Cells:	298
  N Points:	200
  X Bounds:	-3.299e+00, 3.301e+00
  Y Bounds:	-3.294e+00, 3.306e+00
  Z Bounds:	5.400e+00, 5.450e+00
  N Arrays:	3

In [33]:
# check if number of cells are preserved between the original mesh and the new meshes
diff = (bot_mesh.n_cells + tube_bot_mesh.n_cells+ wall_mesh.n_cells + tube_top_mesh.n_cells + top_mesh.n_cells) - mesh.n_cells
if diff != 0:
    raise Exception("cell numbers do not coincide, sum_parts_cells - original_cells = " + str(diff))

In [34]:
# save the 5 parts as new STL files
from pathlib import Path
path = Path(filepath + '/parts')
path.mkdir(parents=True, exist_ok=True)

bot_mesh.save(filepath + '/parts/bot.stl', binary=True)
tube_bot_mesh.save(filepath + '/parts/tube_bot.stl', binary=True)
wall_mesh.save(filepath + '/parts/wall.stl')
tube_top_mesh.save(filepath + '/parts/tube_top.stl', binary=True)
top_mesh.save(filepath + '/parts/top.stl', ascii)
mesh.save(filepath + '/parts/volume.stl', ascii)


### Use meshlab to save the STL files in ASCII format (reomve binary encoding) of 3 parts and volume (initial full STL)


In [35]:
stl_files = [filename for filename in os.listdir(filepath + "/parts") if filename.endswith(".stl")]

In [36]:
for i in stl_files:
    ms_i = pymeshlab.MeshSet()  # create an empty mesh set
    input_path = os.path.join(filepath, "parts", i)  # Full path to the input STL file
    output_path = os.path.join(filepath, "parts", i)  # Full path to the output STL file

    ms_i.load_new_mesh(input_path)  # load the binary encoded STL to the mesh set
    ms_i.save_current_mesh(output_path, binary=False, colormode=False)  # save STL without encoding

### Inside the STL file rename the model to be used in OpenFoam (first line should be "solid NAME" and last line "endsolid NAME"


In [37]:
for i in stl_files:
    with open(filepath + "/parts/" + i) as f:
        lines = f.readlines()
        name_without_extension = i.split('.')[0]
        desired_name = name_without_extension.split('\\')[-1]
        lines[0] = "solid " + desired_name + "\n"
        lines[-1] = "endsolid " + desired_name
    with open(filepath + "/parts/" + i, "w") as f:
        f.writelines(lines)

### Find a random point inside the stl file volume to input to OpenFoam snappyHexMeshDict 

In [38]:
# keep the import here or else it does not work
import numpy as np
from stl import mesh

def generate_random_point(min_coords, max_coords):
    return [np.random.uniform(min_coord, max_coord) for min_coord, max_coord in zip(min_coords, max_coords)]

def is_point_inside_mesh(point, mesh):
    ray_origin = np.array(point)
    ray_direction = np.array([1, 0, 0])  # Choose any arbitrary direction

    # Count the number of ray-mesh intersections
    num_intersections = 0
    for triangle in mesh.vectors:
        v1, v2, v3 = triangle
        normal = np.cross(v2 - v1, v3 - v1)
        denominator = np.dot(ray_direction, normal)
        if abs(denominator) > 1e-6:
            t = np.dot(v1 - ray_origin, normal) / denominator
            if t > 0:
                intersection_point = ray_origin + t * ray_direction
                if is_point_inside_triangle(intersection_point, triangle):
                    num_intersections += 1

    return num_intersections % 2 == 1

def is_point_inside_triangle(point, triangle):
    v1, v2, v3 = triangle
    u = v2 - v1
    v = v3 - v1
    n = np.cross(u, v)
    w = point - v1
    gamma = np.dot(np.cross(u, w), n) / np.dot(n, n)
    beta = np.dot(np.cross(w, v), n) / np.dot(n, n)
    alpha = 1 - gamma - beta
    return 0 <= alpha <= 1 and 0 <= beta <= 1 and 0 <= gamma <= 1

# Load the STL file
stl_file_path = filepath + '/parts/volume.stl'
mesh = mesh.Mesh.from_file(stl_file_path)

# Calculate the bounding box
min_coords = np.min(mesh.vectors, axis=(0, 1))
max_coords = np.max(mesh.vectors, axis=(0, 1))

# Generate a random point inside the volume
point_inside = None
while not point_inside:
    random_point = generate_random_point(min_coords, max_coords)
    if is_point_inside_mesh(random_point, mesh):
        point_inside = random_point

print("Random point inside the volume:", point_inside)


Random point inside the volume: [-1.8100059561902364, 1.9052311129028414, -3.6064812248568874]


# Find the background limits to search for the volume in snappyHexMeshDict and blockMeshDict

In [39]:
mesh = trimesh.load_mesh(stl_file_path)

# Get the bounds of the mesh to define the volume boundaries
min_bound, max_bound = mesh.bounds

# Calculate all eight points of the cuboid that encloses the volume according to the order in OpenFoam
point0 = min_bound * 1.5
point1 = np.array([max_bound[0], min_bound[1], min_bound[2]]) * 1.5
point2 = np.array([max_bound[0], max_bound[1], min_bound[2]]) * 1.5
point3 = np.array([min_bound[0], max_bound[1], min_bound[2]]) * 1.5
point4 = np.array([min_bound[0], min_bound[1], max_bound[2]]) * 1.5
point5 = np.array([max_bound[0], min_bound[1], max_bound[2]]) * 1.5
point6 = max_bound * 1.5
point7 = np.array([min_bound[0], max_bound[1], max_bound[2]]) * 1.5

### Save the specified variables (Cutting specifications)

In [40]:
# Variables to save
Specifications_type = "OpenFoam_snappyHexMeshDict_&_blockMeshDict"

# Convert NumPy arrays to strings
point0_str = str(point0) + ' <-- MIN'
point6_str = str(point6) + ' <-- MAX'

# Combine variables into a matrix
specifications = {
    "height of the actual model": height_range,
    "\n\nfull_volume_mesh": mesh,
    "top_mesh": top_mesh,
    "wall_mesh": wall_mesh,
    "bot_mesh": bot_mesh,
    "diff_num_cell_(if not 0 then there is a problem)": diff,
    "\n\nSpecifications": Specifications_type,
    "Random point inside the volume": point_inside,
    "cuboid enclosing sandwiched model Point 0": point0_str,
    "cuboid enclosing sandwiched model Point 1": point1,
    "cuboid enclosing sandwiched model Point 2": point2,
    "cuboid enclosing sandwiched model Point 3": point3,
    "cuboid enclosing sandwiched model Point 4": point4,
    "cuboid enclosing sandwiched model Point 5": point5,
    "cuboid enclosing sandwiched model Point 6": point6_str,
    "cuboid enclosing sandwiched model Point 7": point7,
}

# create (overwrite) the file in append mode
with open(Specifications_file_path, 'a') as file:
    # Write each variable as "variable_name: variable"
    for name, value in specifications.items():
        file.write(f"{name}: {value}\n")
